In [14]:
import pandas as pd

df = pd.read_csv('../data/processed/stocks_daily.csv', parse_dates=['Date'])
df

,Date,Symbol,Adj Close,Close,High,Low,Open,Volume,corp_action_break,Company Name,Industry,nifty_open,nifty_high,nifty_low,nifty_close,nifty_adj_close,nifty_volume
0,2010-01-04,ADANIPORTS,101.437126,111.349998,112.559998,110.699997,111.000000,1079040,False,Adani Ports and Special Economic Zone Ltd.,SERVICES,5200.899902,5238.450195,5167.100098,5232.200195,5232.200195,0.0
1,2010-01-05,ADANIPORTS,104.935226,115.190002,116.699997,111.800003,111.959999,2112500,False,Adani Ports and Special Economic Zone Ltd.,SERVICES,5277.149902,5288.350098,5242.399902,5277.899902,5277.899902,0.0
2,2010-01-06,ADANIPORTS,110.136932,120.900002,122.000000,113.070000,115.800003,5761450,False,Adani Ports and Special Economic Zone Ltd.,SERVICES,5278.149902,5310.850098,5260.049805,5281.799805,5281.799805,0.0
3,2010-01-07,ADANIPORTS,108.834229,119.470001,123.209999,119.000000,121.489998,3174260,False,Adani Ports and Special Economic Zone Ltd.,SERVICES,5281.799805,5302.549805,5244.750000,5263.100098,5263.100098,0.0
4,2010-01-08,ADANIPORTS,108.706688,119.330002,121.699997,118.639999,119.400002,1220560,False,Adani Ports and Special Economic Zone Ltd.,SERVICES,5264.250000,5276.750000,5234.700195,5244.750000,5244.750000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196774,2026-08-19,ZEEL,102.220001,102.220001,105.879997,101.250000,105.309998,15207062,False,Zee Entertainment Enterprises Ltd.,MEDIA & ENTERTAINMENT,24152.050781,24172.849609,24025.650391,24078.300781,24078.300781,238700.0
196775,2026-08-20,ZEEL,108.029999,108.029999,109.489998,102.750000,103.199997,45642965,False,Zee Entertainment Enterprises Ltd.,MEDIA & ENTERTAINMENT,24225.449219,24265.150391,24184.550781,24231.849609,24231.849609,253900.0
196776,2026-08-21,ZEEL,107.580002,107.580002,109.430000,106.519997,108.500000,23307195,False,Zee Entertainment Enterprises Ltd.,MEDIA & ENTERTAINMENT,24284.050781,24284.050781,24206.800781,24252.000000,24252.000000,259300.0
196777,2026-08-24,ZEEL,104.570000,104.570000,110.629997,103.620003,107.980003,36707649,False,Zee Entertainment Enterprises Ltd.,MEDIA & ENTERTAINMENT,24285.050781,24313.000000,24144.300781,24219.050781,24219.050781,236300.0


In [15]:
# 1. Check for null values
null_counts = df.isnull().sum()
print("Null values per column:")
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "No null values found.")

Null values per column:
No null values found.


In [16]:
# 2. Handle null values (if any)
# Forward-fill price/volume columns within each stock separately (never
# across stocks), then drop any rows still missing critical fields.
price_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
df[price_cols] = df.groupby('Symbol')[price_cols].transform(lambda s: s.ffill())

critical_cols = ['Date', 'Symbol', 'Close']
before = len(df)
df = df.dropna(subset=critical_cols)
after = len(df)
print(f"Dropped {before - after} rows with missing critical values.")

remaining = df.isnull().sum()
print("Remaining nulls:")
print(remaining[remaining > 0] if remaining.sum() > 0 else "None")

Dropped 0 rows with missing critical values.
Remaining nulls:
None


In [17]:
# 3. Sort data for processing (Date-wise, then Symbol for ties)
df = df.sort_values(['Date', 'Symbol']).reset_index(drop=True)
df.head()

,Date,Symbol,Adj Close,Close,High,Low,Open,Volume,corp_action_break,Company Name,Industry,nifty_open,nifty_high,nifty_low,nifty_close,nifty_adj_close,nifty_volume
0,2010-01-04,ADANIPORTS,101.437126,111.349998,112.559998,110.699997,111.000000,1079040,False,Adani Ports and Special Economic Zone Ltd.,SERVICES,5200.899902,5238.450195,5167.100098,5232.200195,5232.200195,0.0
1,2010-01-04,ASIANPAINT,153.844940,178.865005,179.990005,178.000000,179.100006,80350,False,Asian Paints Ltd.,CONSUMER GOODS,5200.899902,5238.450195,5167.100098,5232.200195,5232.200195,0.0
2,2010-01-04,AXISBANK,180.488739,198.419998,199.990005,197.619995,198.779999,4371510,False,Axis Bank Ltd.,FINANCIAL SERVICES,5200.899902,5238.450195,5167.100098,5232.200195,5232.200195,0.0
3,2010-01-04,BAJAJ-AUTO,570.148438,866.025024,886.474976,862.525024,882.500000,381510,False,Bajaj Auto Ltd.,AUTOMOBILE,5200.899902,5238.450195,5167.100098,5232.200195,5232.200195,0.0
4,2010-01-04,BAJAJFINSV,34.690044,35.314632,35.491478,34.017780,34.332169,2469924,False,Bajaj Finserv Ltd.,FINANCIAL SERVICES,5200.899902,5238.450195,5167.100098,5232.200195,5232.200195,0.0


In [18]:
# 4. Drop redundant columns
# - nifty_adj_close: identical to nifty_close for every row (index has no
#   dividend adjustment), pure duplicate.
# - Company Name: 1:1 with Symbol, adds nothing as a model feature (kept in
#   stock_metadata.csv for display/lookup purposes instead).
df = df.drop(columns=['nifty_adj_close', 'Company Name'])
df.head()

,Date,Symbol,Adj Close,Close,High,Low,Open,Volume,corp_action_break,Industry,nifty_open,nifty_high,nifty_low,nifty_close,nifty_volume
0,2010-01-04,ADANIPORTS,101.437126,111.349998,112.559998,110.699997,111.000000,1079040,False,SERVICES,5200.899902,5238.450195,5167.100098,5232.200195,0.0
1,2010-01-04,ASIANPAINT,153.844940,178.865005,179.990005,178.000000,179.100006,80350,False,CONSUMER GOODS,5200.899902,5238.450195,5167.100098,5232.200195,0.0
2,2010-01-04,AXISBANK,180.488739,198.419998,199.990005,197.619995,198.779999,4371510,False,FINANCIAL SERVICES,5200.899902,5238.450195,5167.100098,5232.200195,0.0
3,2010-01-04,BAJAJ-AUTO,570.148438,866.025024,886.474976,862.525024,882.500000,381510,False,AUTOMOBILE,5200.899902,5238.450195,5167.100098,5232.200195,0.0
4,2010-01-04,BAJAJFINSV,34.690044,35.314632,35.491478,34.017780,34.332169,2469924,False,FINANCIAL SERVICES,5200.899902,5238.450195,5167.100098,5232.200195,0.0


In [19]:
df.shape

(196779, 15)

In [20]:
df.head()

,Date,Symbol,Adj Close,Close,High,Low,Open,Volume,corp_action_break,Industry,nifty_open,nifty_high,nifty_low,nifty_close,nifty_volume
0,2010-01-04,ADANIPORTS,101.437126,111.349998,112.559998,110.699997,111.000000,1079040,False,SERVICES,5200.899902,5238.450195,5167.100098,5232.200195,0.0
1,2010-01-04,ASIANPAINT,153.844940,178.865005,179.990005,178.000000,179.100006,80350,False,CONSUMER GOODS,5200.899902,5238.450195,5167.100098,5232.200195,0.0
2,2010-01-04,AXISBANK,180.488739,198.419998,199.990005,197.619995,198.779999,4371510,False,FINANCIAL SERVICES,5200.899902,5238.450195,5167.100098,5232.200195,0.0
3,2010-01-04,BAJAJ-AUTO,570.148438,866.025024,886.474976,862.525024,882.500000,381510,False,AUTOMOBILE,5200.899902,5238.450195,5167.100098,5232.200195,0.0
4,2010-01-04,BAJAJFINSV,34.690044,35.314632,35.491478,34.017780,34.332169,2469924,False,FINANCIAL SERVICES,5200.899902,5238.450195,5167.100098,5232.200195,0.0


In [22]:
df1=pd.read_csv('../data/processed/splits/train.csv', parse_dates=['Date'])

In [23]:
df1

,Date,Symbol,Industry,Adj Close,Close,log_return,price_to_sma10,price_to_sma20,price_to_sma50,price_to_ema20,...,fwd_return_60d,fwd_return_90d,rel_return_7d,rel_return_30d,rel_return_60d,rel_return_90d,beats_median_7d,beats_median_30d,beats_median_60d,beats_median_90d
0,2016-05-30,ADANIPORTS,SERVICES,180.292007,190.250000,-0.391021,1.109722,-0.172081,-1.637088,-0.465406,...,0.419711,0.402628,0.054181,0.078945,0.302243,0.264361,1.0,1.0,1.0,1.0
1,2016-05-30,ASIANPAINT,CONSUMER GOODS,900.225342,984.900024,-0.674579,0.350772,0.878764,1.019553,0.830664,...,0.142066,0.209181,0.018719,-0.025291,0.024598,0.070913,1.0,0.0,1.0,1.0
2,2016-05-30,AXISBANK,FINANCIAL SERVICES,500.884674,513.700012,0.098081,0.667033,0.838140,1.150472,0.921422,...,0.146302,0.048943,0.034312,0.035373,0.028834,-0.089324,1.0,1.0,1.0,0.0
3,2016-05-30,BAJAJ-AUTO,AUTOMOBILE,2017.413208,2607.149902,0.787417,1.393852,0.816122,0.610475,0.939505,...,0.102779,0.108082,-0.005273,-0.049790,-0.014689,-0.030185,0.0,0.0,0.0,0.0
4,2016-05-30,BAJAJFINSV,FINANCIAL SERVICES,183.052673,183.914993,0.066391,0.230018,-0.238450,0.089535,-0.022053,...,0.474268,0.712666,-0.003494,0.253857,0.356800,0.574399,0.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68196,2022-08-23,ULTRACEMCO,CEMENT & CEMENT PRODUCTS,6259.476562,6495.399902,0.183608,-0.537198,-0.327953,0.786817,-0.079916,...,0.039944,0.080449,0.009469,-0.035140,0.011268,0.048645,1.0,0.0,1.0,1.0
68197,2022-08-23,UPL,FERTILISERS & PESTICIDES,732.627930,759.549988,0.929355,-0.453574,0.085308,0.887309,0.221809,...,0.000066,-0.049437,-0.021274,-0.099928,-0.028610,-0.081242,0.0,0.0,0.0,0.0
68198,2022-08-23,VEDL,METALS,168.681427,262.250000,0.980613,0.090869,0.479376,1.278843,0.693420,...,0.174070,0.277317,-0.022237,0.111872,0.145394,0.245513,0.0,1.0,1.0,1.0
68199,2022-08-23,WIPRO,IT,192.194611,208.324997,-0.411090,-1.132393,-0.617726,-0.306514,-0.693298,...,-0.067203,-0.056642,-0.024652,-0.018695,-0.095879,-0.088447,0.0,0.0,0.0,0.0


---
# NSE 200 Results
Generated by the Phase 6 / 8 / 9 / 10 pipeline runs. Run the cells below (paths are relative to `src/`).

In [ ]:
# === NSE200 RESULTS: dataset ===
import pandas as pd, numpy as np
pd.set_option('display.width', 160); pd.set_option('display.max_rows', 80)

for f in ['../data/processed/model_features.csv',
          '../data/processed/splits/train.csv',
          '../data/processed/splits/val.csv',
          '../data/processed/splits/test.csv']:
    d = pd.read_csv(f, usecols=[0, 1])
    print(f'{f.split("/")[-1]:20s} {len(d):>9,} rows  {d.iloc[:,1].nunique():>3} symbols  '
          f'{d.iloc[:,0].min()} -> {d.iloc[:,0].max()}')

In [ ]:
# === NSE200 RESULTS: Phase 6 - directional accuracy (test = 59k rows, base rate ~0.50) ===
p6 = pd.read_csv('../data/phase6_baseline_results.csv')
acc = (p6[p6.task == 'classification']
       .pivot_table(index=['target_set', 'horizon'], columns='model', values='Accuracy')
       .round(4))
acc.columns = [c.split('_', 1)[1] for c in acc.columns]
acc

In [ ]:
# === NSE200 RESULTS: Phase 9 - hybrid stacking (does combining models help?) ===
p9 = pd.read_csv('../data/phase9_hybrid_results.csv')
p9['best_single'] = p9[['base_logit', 'base_rf', 'base_hgb']].max(axis=1)
p9['stack_beats_best?'] = np.where(p9.stacked_meta > p9.best_single, 'yes', 'NO')
p9[['target', 'horizon', 'base_rate', 'best_single', 'blend_avg', 'stacked_meta', 'stack_beats_best?']].round(4)

In [ ]:
# === NSE200 RESULTS: Phase 8 - feature IC stability across 10 years ===
p8 = pd.read_csv('../data/phase8_ic_decay_summary.csv', index_col=0)
p8 = p8[['mean_ic', 'std_ic', 'sign_flips', 'stability_score']].round(3).sort_values('stability_score', ascending=False)
print('MOST stable (low sign_flips, high score):'); display(p8.head(6))
print('\nLEAST stable (regime-dependent - flip sign 5+ of 10 years):'); display(p8[p8.sign_flips >= 5])

In [ ]:
# === NSE200 RESULTS: Phase 10 - portfolio backtest (the headline result) ===
p10 = pd.read_csv('../data/phase10_portfolio_results.csv').set_index('model').round(4)
p10.T

In [ ]:
# === NSE200 RESULTS: Phase 10 - equity curves ===
import matplotlib.pyplot as plt
cur = pd.read_csv('../data/phase10_portfolio_curve_logit.csv', parse_dates=['date'])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(cur.date, cur.long_equity,  label='Long top-quintile (net cost)', lw=2)
ax[0].plot(cur.date, cur.bench_equity, label='Benchmark (equal-weight 200)', lw=2)
ax[0].plot(cur.date, cur.ls_equity,    label='Long-Short (market-neutral)', lw=1.5, ls='--')
ax[0].set_title('Growth of 1.0 (logit, 60d rebalance, 2021-2026)'); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].bar(range(len(cur)), cur.IC, color=np.where(cur.IC >= 0, '#2a9d8f', '#e76f51'))
ax[1].axhline(cur.IC.mean(), color='k', ls='--', label=f'mean IC = {cur.IC.mean():+.3f}')
ax[1].set_title('Per-rebalance Information Coefficient (Spearman)'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"Long book : total return {cur.long_equity.iloc[-1]-1:+.1%} over {len(cur)} periods")
print(f"Benchmark : total return {cur.bench_equity.iloc[-1]-1:+.1%}")
print(f"Long beat benchmark in {(cur.long_net > cur.benchmark).mean():.0%} of periods")